In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from pydantic import BaseModel,Field

In [2]:
load_dotenv()

True

In [ ]:
llm=ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

Hello! I'm doing well, thank you. How are you doing today? How can I help you?


In [10]:
class Feedback(BaseModel):
    sentiment:Literal['positive','negative']
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [11]:
model1=llm.with_structured_output(Feedback)
model2=llm.with_structured_output(DiagnosisSchema)

In [15]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal['positive','negative']
    response:str
    diagnosis:dict

In [40]:
def find_sentiment(state:ReviewState):
    prompt=f"For the following find the sentiment for the below review :\n {state['review']}"
    output=model1.invoke(prompt)
    
    return {"sentiment":output.sentiment}

def run_diagnosis(state:ReviewState):
    prompt=f"For the following review, find out the issue type, tone and urgency:\n {state['review']}"
    output=model2.invoke(prompt)
    
    return {"diagnosis":output.model_dump()}

def negative_response(state:ReviewState):
    diagnosis=state['diagnosis']
    prompt=f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
    """
    output=llm.invoke(prompt).content[0]['text']
    return {"response":output}

def positive_response(state:ReviewState):
    prompt=f"""You are a support assistant agent. Customer posted a postive response. Write a positive message him and ask for feedback"""
    output=llm.invoke(prompt).content[0]['text']
    return {"response":output}
def check_sentiment(state:ReviewState)->Literal['positive_response','run_diagnosis']:
    if(state['sentiment']=='negative'):
        return "run_diagnosis"
    return "positive_response"

In [41]:
graph=StateGraph(ReviewState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("positive_response",positive_response)
graph.add_node("negative_response",negative_response)

graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges("find_sentiment",check_sentiment)
graph.add_edge("run_diagnosis","negative_response")
graph.add_edge("positive_response",END)
graph.add_edge("negative_response",END)


workflow=graph.compile()

In [43]:
initial_state={"review":"The product extremely perfect"}
final_state=workflow.invoke(initial_state)
print(final_state)

{'review': 'The product extremely perfect', 'sentiment': 'positive', 'response': "Hi there! \n\nThank you so much for the wonderful feedback! It truly made our day to hear that you had a great experience with us. We always strive to provide the best possible support, so knowing we hit the mark with you is fantastic. \n\nSince you’ve had such a positive experience, would you mind taking just a quick moment to leave us a review or share a bit more about what you loved? Your insights not only help our team grow, but they also help other customers know what to expect. \n\nHere is a quick link where you can share your thoughts: [Insert Link Here]\n\nThanks again for choosing us, and please don't hesitate to reach out if you ever need anything else! \n\nBest regards,\n\n[Your Name/Company Name] Support Team"}
